# Tripwire: train the tool-output injection classifiers (Colab / Kaggle T4)

Trains **MiniLM-L6** (default candidate) and **DeBERTa-v3-small** (accuracy reference) on the same dataset and splits (seed 42). It exports int8 ONNX artifacts in the gateway's format, evaluates them exactly as the gateway runs them, and optionally evaluates **Llama Prompt Guard 2 22M** zero-shot.

1. Runtime → Change runtime type → **T4 GPU** (Kaggle: Accelerator → GPU T4). Python 3.12 or newer (Colab currently runs 3.13).
2. **Run all.** Every step stops the notebook with a clear `STEP FAILED` error if it fails, so later cells never run on missing output. The preflight cell exports a tiny model in seconds, so export problems show up before any training.
3. The last cell checks every artifact against the gateway contract, prints a summary table and writes **`tripwire-artifacts.zip`**. Download it and send it back.

Rough time on a free T4: dataset about 5 min, MiniLM selection and training about 15 min, DeBERTa-v3-small training about 35 min. Evaluation runs the int8 ONNX files on CPU one window at a time (exactly like the gateway): about 5 min for MiniLM and 30–40 min for DeBERTa.

Prompt Guard 2 is gated (Llama 4 Community License). Request access on its Hugging Face page and add an `HF_TOKEN` secret (Colab: key icon; Kaggle: Add-ons → Secrets), or set `EVAL_PROMPT_GUARD = False`. Its weights are evaluated only; they are never zipped or published.

In [ ]:
import os, subprocess, sys, json, pathlib

def sh(cmd: str, step: str) -> None:
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"STEP FAILED ({step}): `{cmd}` exited with {p.returncode}. Fix the error above and re-run from this cell.")

def require(paths, step: str) -> None:
    missing = [str(p) for p in paths if not pathlib.Path(p).exists()]
    if missing:
        raise RuntimeError(f"STEP FAILED ({step}): expected output missing: {missing}")

print("Python", sys.version.split()[0])
if sys.version_info < (3, 12):
    raise RuntimeError("STEP FAILED (runtime): Python 3.12+ required (pinned numpy needs it)")
sh("nvidia-smi -L || echo 'no GPU: training will be slow'", "gpu check")
if not pathlib.Path("scripts/preflight.py").exists():  # re-running this cell is safe
    if not pathlib.Path("tripwire-agent-gateway").exists():
        sh("git clone --depth 1 https://github.com/yugpatill/tripwire-agent-gateway.git", "clone")
    os.chdir("tripwire-agent-gateway/ml")
sh("git log -1 --oneline", "clone")

In [ ]:
# 1. Remove preinstalled packages that need huggingface_hub>=1 (our transformers pin needs <1.0);
#    optimum's exporter imports diffusers when it is present and fails.
sh("pip uninstall -y -q diffusers gradio gradio-client", "uninstall conflicting packages")
# 2. Exact pins (same versions as ml/uv.lock). torch is not in the file: the runtime's CUDA build is kept.
sh("pip install -q -r requirements-colab.txt", "install pinned requirements")

In [ ]:
# Fails in seconds if anything is off: Python, torch range, pins, conflicting packages,
# imports, and a tiny BERT + DeBERTa ONNX export, int8 quantization and run.
sh("python scripts/preflight.py", "preflight")

In [ ]:
EVAL_PROMPT_GUARD = True   # zero-shot baseline; needs HF_TOKEN with approved access

try:
    from google.colab import userdata  # Colab
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
if EVAL_PROMPT_GUARD and not os.environ.get("HF_TOKEN"):
    print("Prompt Guard 2 will be skipped (no HF_TOKEN). Set EVAL_PROMPT_GUARD = False to silence this.")

In [ ]:
sh("python scripts/build_dataset.py", "build dataset")
require([f"data/{s}.jsonl" for s in ("train", "val", "test", "test_ood")], "build dataset")
stats = json.loads(pathlib.Path("data/stats.json").read_text())
for split, s in stats.items():
    if s["_total"] == 0 or s["_injection"] == 0:
        raise RuntimeError(f"STEP FAILED (build dataset): split {split} is empty or has no injections")
print({k: (v["_total"], v["_injection"]) for k, v in stats.items()})

### Pick the MiniLM-L6 checkpoint
Both candidates have the same architecture (6 layers, hidden size 384, 512 positions), so their latency is identical:
* `nreimers/MiniLM-L6-H384-uncased` (MIT): a general-purpose distilled language model, the usual starting point for classification.
* `sentence-transformers/all-MiniLM-L6-v2` (Apache-2.0): the same backbone further tuned for sentence embeddings on inputs of up to 256 tokens.

Each gets one epoch on the same data; the higher validation PR-AUC wins and is trained fully in the next cell.

In [ ]:
CANDIDATES = {"minilm-l6": "nreimers/MiniLM-L6-H384-uncased", "minilm-l6-st": "sentence-transformers/all-MiniLM-L6-v2"}
selection = {}
for key, hf_id in CANDIDATES.items():
    sh(f"python scripts/train.py --model {key} --epochs 1 --output-key select-{key}", f"MiniLM selection: {key}")
    info_path = pathlib.Path(f"checkpoints/select-{key}/best/train_info.json")
    require([info_path], f"MiniLM selection: {key}")
    selection[hf_id] = json.loads(info_path.read_text())["val"]
best_hf = max(selection, key=lambda k: selection[k]["eval_pr_auc"])
print(json.dumps(selection, indent=2)); print("selected:", best_hf)
pathlib.Path("artifacts").mkdir(exist_ok=True)
pathlib.Path("artifacts/minilm_selection.json").write_text(json.dumps({"candidates": selection, "selected": best_hf}, indent=2))
sh("rm -rf checkpoints/select-*", "MiniLM selection cleanup")

In [ ]:
def train_export_evaluate(key: str, extra: str = "") -> None:
    art = pathlib.Path(f"artifacts/injection-{key}")
    sh(f"python scripts/train.py --model {key} {extra}", f"train {key}")
    require([f"checkpoints/{key}/best/train_info.json"], f"train {key}")
    sh(f"python scripts/export_onnx.py --model {key}", f"export {key}")
    require([art / "model.int8.onnx", art / "tokenizer.json", art / "config.json"], f"export {key}")
    sh(f"python scripts/evaluate.py --model {key} --threads 2", f"evaluate {key}")
    require([art / "metrics.json"], f"evaluate {key}")

train_export_evaluate("minilm-l6", f"--hf-id {best_hf}")

In [ ]:
train_export_evaluate("deberta-v3-small")

In [ ]:
if EVAL_PROMPT_GUARD and os.environ.get("HF_TOKEN"):
    art = pathlib.Path("artifacts/injection-prompt-guard-2-22m")
    sh("python scripts/export_onnx.py --model prompt-guard-2-22m", "export prompt-guard-2-22m (check that access is approved on Hugging Face)")
    require([art / "model.int8.onnx", art / "config.json"], "export prompt-guard-2-22m")
    sh("python scripts/evaluate.py --model prompt-guard-2-22m --threads 2", "evaluate prompt-guard-2-22m")
    require([art / "metrics.json"], "evaluate prompt-guard-2-22m")
else:
    print("Prompt Guard 2 skipped.")

In [ ]:
# Contract check (model.int8.onnx, tokenizer.json, config.json {max_length, threshold, labels},
# metrics.json; loads and scores in onnxruntime), summary table, then the zip.
sh("python scripts/check_artifacts.py --require minilm-l6,deberta-v3-small --zip tripwire-artifacts.zip", "artifact contract check")
require(["tripwire-artifacts.zip"], "zip")
try:
    from google.colab import files
    files.download("tripwire-artifacts.zip")
except Exception:
    print("Download tripwire-artifacts.zip from the file browser (Kaggle: /kaggle/working/tripwire-agent-gateway/ml).")